# Bibliotecas

In [5]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path
from IPython.display import display

# Configurações

In [6]:
DATA_DIR = Path("../data")

PATH_GEOINFO = DATA_DIR / "processed/geoinfo_artigos_processed.csv"
PATH_GOOGLE = DATA_DIR / "raw/citacoes_google_scholar.csv"
PATH_OPENALEX = DATA_DIR / "raw/citacoes_openalex.csv"

OUTPUT_DIR = DATA_DIR / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
COLUNAS_GEOINFO = [
    "titulo",
    "ano",
    "autores",
    "instituicoes",
    "edicao",
    "identificador",
    "url_edicao",
    "url_artigo",
    "url_metadata",
    "numero_edicao"
]

COLUNAS_CITANTES = [
    "titulo_original",
    "ano_original",
    "titulo",
    "ano",
    "autores",
    "instituicoes",
    "idioma",
    "pais",
    "veiculo_publicacao",
    "doi",
    "url"
]

COLUNAS_OPENALEX = [
    *COLUNAS_CITANTES,
    "tipo_documento",
    "topico",
    "subcampo",
    "campo",
    "dominio_tematico",
    "fonte_publicacao",
    "status_acesso_aberto",
    "openalex_id"
]

COLUNAS_TEXTO = [
    "autores",
    "instituicoes",
    "idioma",
    "pais",
    "veiculo_publicacao",
    "tipo_documento",
    "topico",
    "subcampo",
    "campo",
    "dominio_tematico",
    "fonte_publicacao"
]

COLUNAS_CONSOLIDADAS = [
    "id_geoinfo",
    "id_citante",
    "titulo",
    "titulo_padronizado",
    "ano",
    "autores",
    "instituicoes",
    "idioma_padronizado",
    "pais_padronizado",
    "veiculo_publicacao",
    "doi_normalizado",
    "url",
    "tipo_documento",
    "topico",
    "subcampo",
    "campo",
    "dominio_tematico",
    "fonte_publicacao",
    "status_acesso_aberto",
    "openalex_id"
]

# Carregamento dos dados

In [8]:
df_geoinfo = pd.read_csv(PATH_GEOINFO, encoding="utf-8")
df_google = pd.read_csv(PATH_GOOGLE, encoding="utf-8")
df_openalex = pd.read_csv(PATH_OPENALEX, encoding="utf-8")

In [9]:
print(f"GEOINFO:       {df_geoinfo.shape[0]:,} registros")
print(f"Google Scholar:{df_google.shape[0]:,} registros")
print(f"OpenAlex:      {df_openalex.shape[0]:,} registros")

GEOINFO:       681 registros
Google Scholar:2,988 registros
OpenAlex:      0 registros


In [10]:
df_geoinfo.head(3)

,titulo,ano,autores,instituicoes,edicao,identificador,idioma,url_edicao,url_artigo,url_metadata,numero_edicao
0,Reproducible and empirical method refines Frac...,2025,"1 Adorno, Bruno Vargas 2 Nesbitt, Lorien 3 Ama...",1 National Institute for Space Research (INPE)...,25a. Edição São José dos Campos 2025,8JMKD2USPTW34P/4DKBAQH,en,http://urlib.net/ibi/8JMKD2USPTW34P/4DKC4FB,http://mtc-m16c.sid.inpe.br/col/sid.inpe.br/mt...,http://mtc-m16c.sid.inpe.br/sid.inpe.br/mtc-m1...,25
1,Potential Effects of Legal Reserve Exclusion o...,2025,"1 Andrade, Pedro Ribeiro 2 Rodrigues, Erick Te...",1 National Institute for Space Research (INPE)...,25a. Edição São José dos Campos 2025,8JMKD2USPTW34P/4DKBE9S,en,http://urlib.net/ibi/8JMKD2USPTW34P/4DKC4FB,http://mtc-m16c.sid.inpe.br/col/sid.inpe.br/mt...,http://mtc-m16c.sid.inpe.br/sid.inpe.br/mtc-m1...,25
2,Bill 191/2020: Illegal Mining and Land Use Lan...,2025,"1 Chuizaca-Espinoza, Isabel Adriana 2 Amaral, ...",1 National Institute for Space Research (INPE)...,25a. Edição São José dos Campos 2025,8JMKD2USPTW34P/4DKBSAB,en,http://urlib.net/ibi/8JMKD2USPTW34P/4DKC4FB,http://mtc-m16c.sid.inpe.br/col/sid.inpe.br/mt...,http://mtc-m16c.sid.inpe.br/sid.inpe.br/mtc-m1...,25


In [11]:
df_google.head(3)

,titulo_original,ano_original,titulo,ano,autores,instituicoes,idioma,pais,veiculo_publicacao,tipo_documento,doi,url
0,Predicting Oceanic Wind Speed and Direction Us...,2025,Modelando Ocorrências de Espécies Marinhas com...,2026.0,"H Margotte, CS Hara, ATR Pozo - Anais do Compu...",NaN,NaN,NaN,periodicos.univali.br,NaN,NaN,https://periodicos.univali.br/index.php/acotb/...
1,Implementing a new automatic deforestation mon...,2025,Beyond the reporting of disturbed areas: the u...,2026.0,"W Leal Filho, FC Alves, MS Reis, VL Camilotti…...",NaN,NaN,NaN,Springer,NaN,NaN,https://link.springer.com/article/10.1186/s405...
2,Implementing a new automatic deforestation mon...,2025,Configuration Assessment of Deter-RT: a New SA...,2025.0,"MS Reis, J Doblas, LH Gusmão… - Rev. Bras …, 2025",NaN,NaN,NaN,researchgate.net,NaN,NaN,https://www.researchgate.net/profile/Mariane-R...


In [12]:
df_openalex.head(3)

,titulo_original,ano_original,url_original,titulo,ano,autores,instituicoes,pais,idioma,doi,tipo_documento,topico,subcampo,campo,dominio_tematico,fonte_publicacao,status_acesso_aberto,openalex_id,url


In [13]:
print("Colunas GEOINFO:")
print(df_geoinfo.columns.tolist())

print("\nColunas Google Scholar:")
print(df_google.columns.tolist())

print("\nColunas OpenAlex:")
print(df_openalex.columns.tolist())

Colunas GEOINFO:
['titulo', 'ano', 'autores', 'instituicoes', 'edicao', 'identificador', 'idioma', 'url_edicao', 'url_artigo', 'url_metadata', 'numero_edicao']

Colunas Google Scholar:
['titulo_original', 'ano_original', 'titulo', 'ano', 'autores', 'instituicoes', 'idioma', 'pais', 'veiculo_publicacao', 'tipo_documento', 'doi', 'url']

Colunas OpenAlex:
['titulo_original', 'ano_original', 'url_original', 'titulo', 'ano', 'autores', 'instituicoes', 'pais', 'idioma', 'doi', 'tipo_documento', 'topico', 'subcampo', 'campo', 'dominio_tematico', 'fonte_publicacao', 'status_acesso_aberto', 'openalex_id', 'url']


In [14]:
def verificar_colunas(df, esperadas, nome, obrigatorio=True):
    faltantes = set(esperadas) - set(df.columns)
    extras = set(df.columns) - set(esperadas)

    if faltantes:
        msg = f"{nome}: colunas ausentes -> {faltantes}"
        if obrigatorio:
            raise ValueError(msg)
        print(msg)
    else:
        print(f"{nome}: estrutura OK")

    if extras:
        print(f"{nome}: colunas extras (não esperadas) -> {extras}")


fontes = [
    (df_geoinfo, COLUNAS_GEOINFO, "GEOINFO"),
    (df_google, COLUNAS_CITANTES, "Google Scholar"),
    (df_openalex, COLUNAS_OPENALEX, "OpenAlex"),
]

for df, esperadas, nome in fontes:
    verificar_colunas(df, esperadas, nome)

GEOINFO: estrutura OK
GEOINFO: colunas extras (não esperadas) -> {'idioma'}
Google Scholar: estrutura OK
Google Scholar: colunas extras (não esperadas) -> {'tipo_documento'}


ValueError: OpenAlex: colunas ausentes -> {'veiculo_publicacao'}

In [15]:
def resumo_nulos(df, nome, incluir_vazios=True):
    if len(df) == 0:
        print(f"\n{nome}: dataframe vazio, nada a resumir")
        return None

    resultado = df.isna().sum().to_frame("nulos")

    if incluir_vazios:
        vazios = df.apply(
            lambda col: col.astype(str).str.strip().eq("").sum()
            if col.dtype == "object" else 0
        )
        resultado["vazios"] = vazios

    resultado["percentual_nulos"] = (resultado["nulos"] / len(df) * 100).round(2)
    resultado = resultado.sort_values("percentual_nulos", ascending=False)

    print(f"\n{nome} ({len(df):,} registros)")
    display(resultado)

    return resultado


fontes = [
    (df_geoinfo, "GEOINFO"),
    (df_google, "Google Scholar"),
    (df_openalex, "OpenAlex"),
]

resumos_nulos = {nome: resumo_nulos(df, nome) for df, nome in fontes}


GEOINFO (681 registros)


,nulos,vazios,percentual_nulos
titulo,0,0,0.0
ano,0,0,0.0
autores,0,0,0.0
instituicoes,0,0,0.0
edicao,0,0,0.0
identificador,0,0,0.0
idioma,0,0,0.0
url_edicao,0,0,0.0
url_artigo,0,0,0.0
url_metadata,0,0,0.0



Google Scholar (2,988 registros)


,nulos,vazios,percentual_nulos
tipo_documento,2988,0,100.00
pais,2988,0,100.00
idioma,2988,0,100.00
instituicoes,2988,0,100.00
doi,2988,0,100.00
ano,418,0,13.99
url,334,0,11.18
veiculo_publicacao,213,0,7.13
titulo,0,0,0.00
ano_original,0,0,0.00



OpenAlex: dataframe vazio, nada a resumir


# Funções auxiliares

In [16]:
VALORES_NULOS_LITERAIS = {"nan", "none", "na", "n/a", "null", "-"}

def normalizar_texto(valor):
    if pd.isna(valor):
        return np.nan

    valor = str(valor)
    valor = valor.replace("\u200b", "").replace("\ufeff", "")  
    valor = valor.strip()
    valor = re.sub(r"\s+", " ", valor)

    if valor == "" or valor.lower() in VALORES_NULOS_LITERAIS:
        return np.nan

    return valor

In [17]:
import html

def normalizar_titulo(valor):
    if pd.isna(valor):
        return np.nan

    valor = html.unescape(str(valor))          
    valor = valor.strip().lower()

    valor = re.sub(r"\s*(\.\.\.|…)\s*$", "", valor)

    valor = unicodedata.normalize("NFKD", valor)
    valor = "".join(c for c in valor if not unicodedata.combining(c))

    valor = re.sub(r"[^\w\s]", " ", valor)

    valor = re.sub(r"\s+", " ", valor).strip()

    return valor if valor else np.nan

# Tratamento e Padronização

In [18]:
def normalizar_colunas_texto(df, colunas=None):
    """Aplica normalizar_texto às colunas de texto informadas (ou a todas object, se colunas=None)."""
    alvo = colunas if colunas is not None else df.select_dtypes(include="object").columns
    for coluna in alvo:
        if coluna in df.columns:
            df[coluna] = df[coluna].apply(normalizar_texto)
    return df

normalizar_colunas_texto(df_geoinfo, ["titulo", "autores", "instituicoes", "edicao"])
normalizar_colunas_texto(df_google, COLUNAS_TEXTO + ["titulo", "titulo_original"])
normalizar_colunas_texto(df_openalex, COLUNAS_TEXTO + ["titulo", "titulo_original"])

,titulo_original,ano_original,url_original,titulo,ano,autores,instituicoes,pais,idioma,doi,tipo_documento,topico,subcampo,campo,dominio_tematico,fonte_publicacao,status_acesso_aberto,openalex_id,url


## Padronização dos títulos

In [19]:
COLUNAS_TITULO = {
    "df_geoinfo": ["titulo"],
    "df_google": ["titulo", "titulo_original"],
    "df_openalex": ["titulo", "titulo_original"],
}

dataframes = {"df_geoinfo": df_geoinfo, "df_google": df_google, "df_openalex": df_openalex}

for nome, colunas in COLUNAS_TITULO.items():
    df = dataframes[nome]
    for coluna in colunas:
        coluna_bruta = f"{coluna}_bruto"
        if coluna_bruta not in df.columns:
            df[coluna_bruta] = df[coluna]  # preserva o valor original antes de normalizar
        df[coluna] = df[coluna].apply(normalizar_titulo)

## Padronizar anos

In [20]:
ANO_MINIMO = 1900
ANO_MAXIMO = pd.Timestamp.now().year + 1  

def padronizar_ano(df, coluna):
    valores_originais = df[coluna].copy()
    convertido = pd.to_numeric(df[coluna], errors="coerce").astype("Int64")

    fora_intervalo = convertido.notna() & ~convertido.between(ANO_MINIMO, ANO_MAXIMO)
    convertido[fora_intervalo] = pd.NA

    perdidos = convertido.isna() & valores_originais.notna()
    if perdidos.sum() > 0:
        print(f"{coluna}: {perdidos.sum()} valores não convertidos ou fora do intervalo -> {valores_originais[perdidos].unique()[:10]}")

    return convertido


for df in [df_geoinfo, df_google, df_openalex]:
    for coluna in ["ano", "ano_original"]:
        if coluna in df.columns:
            df[coluna] = padronizar_ano(df, coluna)

## Padronização do DOI

In [21]:
DOI_VALIDO = re.compile(r"^10\.\d{4,9}/\S+$")

def normalizar_doi(valor):
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip().lower()
    valor = re.sub(r"^https?://(dx\.|www\.)?doi\.org/", "", valor)
    valor = re.sub(r"^doi:\s*", "", valor)
    valor = valor.split("?")[0].split("#")[0]  # remove parâmetros/fragmento
    valor = valor.strip().rstrip("/")

    if valor == "" or not DOI_VALIDO.match(valor):
        return np.nan

    return valor

for df, nome in [(df_google, "Google Scholar"), (df_openalex, "OpenAlex")]:
    antes = df["doi"].notna().sum()
    df["doi_normalizado"] = df["doi"].apply(normalizar_doi)
    depois = df["doi_normalizado"].notna().sum()
    if antes != depois:
        print(f"{nome}: {antes - depois} DOI(s) descartado(s) por formato inválido")

# Integração

In [22]:
def construir_chave(titulo, ano):
    if pd.isna(titulo) or pd.isna(ano):
        return np.nan
    return f"{titulo}|{ano}"


df_geoinfo["chave_geoinfo"] = df_geoinfo.apply(
    lambda row: construir_chave(row["titulo"], row["ano"]), axis=1
)
df_google["chave_geoinfo"] = df_google.apply(
    lambda row: construir_chave(row["titulo_original"], row["ano_original"]), axis=1
)
df_openalex["chave_geoinfo"] = df_openalex.apply(
    lambda row: construir_chave(row["titulo_original"], row["ano_original"]), axis=1
)

duplicados_geoinfo = df_geoinfo["chave_geoinfo"].duplicated(keep=False).sum()
if duplicados_geoinfo:
    print(f"Atenção: {duplicados_geoinfo} registro(s) GEOINFO com chave duplicada — "
          f"citações podem ser atribuídas ao artigo errado")

chaves_geoinfo = set(df_geoinfo["chave_geoinfo"].dropna())

df_google["geoinfo_encontrado"] = df_google["chave_geoinfo"].isin(chaves_geoinfo)
df_openalex["geoinfo_encontrado"] = df_openalex["chave_geoinfo"].isin(chaves_geoinfo)

for df, nome in [(df_google, "Google Scholar"), (df_openalex, "OpenAlex")]:
    print(f"\n{nome}:")
    print(df["geoinfo_encontrado"].value_counts())

    if "titulo_original_bruto" in df.columns:
        truncados = df["titulo_original_bruto"].str.strip().str.endswith(("...", "…"), na=False)
        nao_encontrados = ~df["geoinfo_encontrado"]
        print(f"  não encontrados por título truncado: {(nao_encontrados & truncados).sum()}")
        print(f"  não encontrados por outro motivo:    {(nao_encontrados & ~truncados).sum()}")

display(df_google[~df_google["geoinfo_encontrado"]][["titulo_original", "ano_original"]].drop_duplicates().head(20))
display(df_openalex[~df_openalex["geoinfo_encontrado"]][["titulo_original", "ano_original"]].drop_duplicates().head(20))

mapa_geoinfo = df_geoinfo.drop_duplicates("chave_geoinfo").set_index("chave_geoinfo")["identificador"]

df_google["id_geoinfo"] = df_google["chave_geoinfo"].map(mapa_geoinfo)
df_openalex["id_geoinfo"] = df_openalex["chave_geoinfo"].map(mapa_geoinfo)

df_google["chave_citante"] = df_google.apply(
    lambda row: construir_chave(row["titulo"], row["ano"]), axis=1
)
df_openalex["chave_citante"] = df_openalex.apply(
    lambda row: construir_chave(row["titulo"], row["ano"]), axis=1
)

def criar_chave_citante(df):
    id_citante = df["doi_normalizado"].fillna(df["chave_citante"])
    tipo = np.where(df["doi_normalizado"].notna(), "doi", "titulo_ano")
    return id_citante, tipo

df_google["id_citante"], df_google["id_citante_tipo"] = criar_chave_citante(df_google)
df_openalex["id_citante"], df_openalex["id_citante_tipo"] = criar_chave_citante(df_openalex)

for df, nome in [(df_google, "Google Scholar"), (df_openalex, "OpenAlex")]:
    print(f"\n{nome} — tipo de id_citante:")
    print(df["id_citante_tipo"].value_counts())


Google Scholar:
geoinfo_encontrado
True    2988
Name: count, dtype: int64
  não encontrados por título truncado: 0
  não encontrados por outro motivo:    0

OpenAlex:
Series([], Name: count, dtype: int64)
  não encontrados por título truncado: 0
  não encontrados por outro motivo:    0


,titulo_original,ano_original


,titulo_original,ano_original



Google Scholar — tipo de id_citante:
id_citante_tipo
titulo_ano    2988
Name: count, dtype: int64

OpenAlex — tipo de id_citante:
Series([], Name: count, dtype: int64)


# Autores e Instituições

In [23]:
def tokenizar_por_marcador(valor):
    if pd.isna(valor):
        return []

    tokens = str(valor).strip().split()
    segmentos = []
    ordem_atual = None
    nome_atual = []

    for tok in tokens:
        proximo_esperado = 1 if ordem_atual is None else ordem_atual + 1
        if tok.isdigit() and int(tok) == proximo_esperado:
            if ordem_atual is not None:
                segmentos.append({"ordem": ordem_atual, "texto": " ".join(nome_atual).strip()})
            ordem_atual = int(tok)
            nome_atual = []
        else:
            nome_atual.append(tok)

    if ordem_atual is not None:
        segmentos.append({"ordem": ordem_atual, "texto": " ".join(nome_atual).strip()})

    return segmentos

In [24]:
def extrair_instituicoes_geoinfo(valor):
    segmentos = tokenizar_por_marcador(valor)
    return [
        {"ordem": s["ordem"], "instituicao_original": s["texto"] or None}
        for s in segmentos
    ]

def extrair_autores_geoinfo(valor):
    segmentos = tokenizar_por_marcador(valor)
    segmentos = [s for s in segmentos if s["texto"]]

    autores = []
    i = 0
    while i < len(segmentos):
        texto = segmentos[i]["texto"]
        ordem = segmentos[i]["ordem"]

        if "," not in texto and i + 1 < len(segmentos) and segmentos[i + 1]["texto"].startswith(","):
            texto = f"{texto}{segmentos[i + 1]['texto']}"
            i += 1  

        autores.append({"ordem": ordem, "autor_original": texto.strip()})
        i += 1

    suspeitos = [a for a in autores if a["autor_original"].count(",") != 1]
    if suspeitos:
        print(f"Aviso: {len(suspeitos)} autor(es) sem o padrão 'Sobrenome, Nome' após reconstrução "
              f"-> {[a['autor_original'] for a in suspeitos]} | texto original: {str(valor)[:80]}")

    for nova_ordem, autor in enumerate(autores, start=1):
        autor["ordem"] = nova_ordem

    return autores

In [25]:
def padronizar_nome_autor_geoinfo(nome):
    if pd.isna(nome):
        return np.nan

    nome = str(nome).strip()

    if "," not in nome:
        return nome

    sobrenome, nomes = nome.split(",", 1)
    return f"{nomes.strip()} {sobrenome.strip()}"

def normalizar_para_mapa(valor):
    if pd.isna(valor):
        return np.nan
    valor = unicodedata.normalize("NFKD", str(valor))
    valor = "".join(c for c in valor if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", valor).strip().lower()

In [26]:
registros = []
linhas_inconsistentes = 0

for _, row in df_geoinfo.iterrows():
    autores = extrair_autores_geoinfo(row["autores"])
    instituicoes = extrair_instituicoes_geoinfo(row["instituicoes"])

    inconsistente = bool(autores) and bool(instituicoes) and len(autores) != len(instituicoes)
    if inconsistente:
        linhas_inconsistentes += 1
        print(f"Aviso: {row['identificador']} tem {len(autores)} autor(es) mas "
              f"{len(instituicoes)} instituição(ões) mesmo após correção -> revisar manualmente")

    for i, autor in enumerate(autores):
        instituicao = instituicoes[i]["instituicao_original"] if i < len(instituicoes) else None
        registros.append({
            "id_geoinfo": row["identificador"],
            "ordem": autor["ordem"],
            "autor_original": autor["autor_original"],
            "autor_padronizado": padronizar_nome_autor_geoinfo(autor["autor_original"]),
            "instituicao_original": instituicao,
            "contagem_inconsistente": inconsistente
        })

if linhas_inconsistentes:
    print(f"\nTotal: {linhas_inconsistentes} linha(s) ainda inconsistentes após correção")

df_autores_instituicoes_geoinfo = pd.DataFrame(registros)

Aviso: 8JMKD3MGPDW34P/4ADCBJE tem 5 autor(es) mas 4 instituição(ões) mesmo após correção -> revisar manualmente
Aviso: 8JMKD3MGPDW34P/487MF9E tem 3 autor(es) mas 2 instituição(ões) mesmo após correção -> revisar manualmente
Aviso: 8JMKD3MGPDW34P/43PQSSS tem 3 autor(es) mas 2 instituição(ões) mesmo após correção -> revisar manualmente
Aviso: 8JMKD3MGPDW34P/3SG4CTL tem 5 autor(es) mas 8 instituição(ões) mesmo após correção -> revisar manualmente
Aviso: 1 autor(es) sem o padrão 'Sobrenome, Nome' após reconstrução -> ['Magalhã'] | texto original: 1 Gomes, Thiago L. 2 Magalhã 3 es, Salles V. G. 4 Andrade, Marcus V. A. 5 Pena,
Aviso: 8JMKD3MGPDW34P/3KQ6NNL tem 5 autor(es) mas 4 instituição(ões) mesmo após correção -> revisar manualmente
Aviso: 1 autor(es) sem o padrão 'Sobrenome, Nome' após reconstrução -> ['Cláudio S. Baptista'] | texto original: 1 Almeida, Damiăo R. 2 Cláudio S. Baptista 3 Silva, Elvis R. da 4 Campelo, Cláud
Aviso: 8JMKD3MGP8W/3GQ6SD8 tem 3 autor(es) mas 2 instituição(õe

In [27]:
MAP_INSTITUICOES = {
    "national institute for space research (inpe)": "INPE",
    "instituto nacional de pesquisas espaciais (inpe)": "INPE",
    "instituto nacional de pesquisas espaciais": "INPE",
}

df_autores_instituicoes_geoinfo["_instituicao_chave"] = (
    df_autores_instituicoes_geoinfo["instituicao_original"].apply(normalizar_para_mapa)
)

df_autores_instituicoes_geoinfo["instituicao_padronizada"] = (
    df_autores_instituicoes_geoinfo["_instituicao_chave"]
    .map(MAP_INSTITUICOES)
    .fillna(df_autores_instituicoes_geoinfo["instituicao_original"])
)

df_autores_instituicoes_geoinfo = df_autores_instituicoes_geoinfo.drop(columns="_instituicao_chave")

display(df_autores_instituicoes_geoinfo.head(10))

,id_geoinfo,ordem,autor_original,autor_padronizado,instituicao_original,contagem_inconsistente,instituicao_padronizada
0,8JMKD2USPTW34P/4DKBAQH,1,"Adorno, Bruno Vargas",Bruno Vargas Adorno,National Institute for Space Research (INPE),False,INPE
1,8JMKD2USPTW34P/4DKBAQH,2,"Nesbitt, Lorien",Lorien Nesbitt,University of British Columbia,False,University of British Columbia
2,8JMKD2USPTW34P/4DKBAQH,3,"Amaral, Silvana",Silvana Amaral,National Institute for Space Research (INPE),False,INPE
3,8JMKD2USPTW34P/4DKBE9S,1,"Andrade, Pedro Ribeiro",Pedro Ribeiro Andrade,National Institute for Space Research (INPE),False,INPE
4,8JMKD2USPTW34P/4DKBE9S,2,"Rodrigues, Erick Teixeira",Erick Teixeira Rodrigues,National Institute for Space Research (INPE),False,INPE
5,8JMKD2USPTW34P/4DKBE9S,3,"Simoes, Rolf E. O.",Rolf E. O. Simoes,Open Geo Hub (OGH),False,Open Geo Hub (OGH)
6,8JMKD2USPTW34P/4DKBE9S,4,"Escada, Maria Isabel Sobral",Maria Isabel Sobral Escada,National Institute for Space Research (INPE),False,INPE
7,8JMKD2USPTW34P/4DKBSAB,1,"Chuizaca-Espinoza, Isabel Adriana",Isabel Adriana Chuizaca-Espinoza,National Institute for Space Research (INPE),False,INPE
8,8JMKD2USPTW34P/4DKBSAB,2,"Amaral, Silvana",Silvana Amaral,National Institute for Space Research (INPE),False,INPE
9,8JMKD2USPTW34P/4DKBDA8,1,"Costa, Gabriel F.",Gabriel F. Costa,Federal University of Ouro Preto (UFOP),False,Federal University of Ouro Preto (UFOP)


# Países

In [28]:
MAP_PAISES = {
    "br": "Brasil", "brazil": "Brasil", "brasil": "Brasil",
    "us": "Estados Unidos", "usa": "Estados Unidos",
    "united states": "Estados Unidos", "united states of america": "Estados Unidos",
    "uk": "Reino Unido", "gb": "Reino Unido", "united kingdom": "Reino Unido",
    "it": "Itália", "iq": "Iraque", "cn": "China", "cz": "República Tcheca",
    "ke": "Quênia", "no": "Noruega", "ug": "Uganda", "ca": "Canadá",
    "de": "Alemanha", "ec": "Equador", "in": "Índia",
}

VALORES_PAIS_NULOS = {"nao informado", "na", "n a", "unknown", "-"}

def normalizar_para_mapa_paises(valor):
    valor = unicodedata.normalize("NFKD", str(valor))
    valor = "".join(c for c in valor if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", valor).strip().lower()


def padronizar_pais(valor):
    if pd.isna(valor) or str(valor).strip() == "":
        return "Não informado"

    codigos = [c.strip() for c in re.split(r"[;,]", str(valor)) if c.strip()]
    nomes = []

    for codigo in codigos:
        chave = normalizar_para_mapa_paises(codigo)
        if chave in VALORES_PAIS_NULOS:
            continue
        nomes.append(MAP_PAISES.get(chave, codigo.strip().title()))

    if not nomes:
        return "Não informado"

    return "; ".join(dict.fromkeys(nomes))  #remove os duplicados

df_google["pais_padronizado"] = df_google["pais"].apply(padronizar_pais)
df_openalex["pais_padronizado"] = df_openalex["pais"].apply(padronizar_pais)

for df, nome in [(df_google, "Google Scholar"), (df_openalex, "OpenAlex")]:
    valores_com_fallback = df["pais_padronizado"].str.split("; ").explode()
    nao_mapeados = valores_com_fallback[
        ~valores_com_fallback.isin(list(MAP_PAISES.values()) + ["Não informado"])
    ].value_counts()
    if len(nao_mapeados):
        print(f"\n{nome}: código(s) de país ainda sem mapeamento -> \n{nao_mapeados}")

# Idioma

In [29]:
MAP_IDIOMAS = {
    "en": "Inglês", "eng": "Inglês", "english": "Inglês", "ingles": "Inglês",
    "pt": "Português", "por": "Português", "portuguese": "Português", "portugues": "Português",
    "pt-br": "Português", "pt br": "Português",
    "es": "Espanhol", "spa": "Espanhol", "spanish": "Espanhol", "espanhol": "Espanhol",
}

VALORES_IDIOMA_NULOS = {"nao informado", "na", "n a", "unknown", "-", "not informed"}

def normalizar_para_mapa_idiomas(valor):
    """Normaliza texto só para efeito de casar com MAP_IDIOMAS (não altera o valor exibido)."""
    valor = unicodedata.normalize("NFKD", str(valor))
    valor = "".join(c for c in valor if not unicodedata.combining(c))
    valor = valor.replace("-", " ")
    return re.sub(r"\s+", " ", valor).strip().lower()


def padronizar_idioma(valor):
    if pd.isna(valor) or str(valor).strip() == "":
        return "Não informado"

    chave = normalizar_para_mapa_idiomas(valor)

    if chave in VALORES_IDIOMA_NULOS:
        return "Não informado"

    return MAP_IDIOMAS.get(chave, str(valor).strip().title())


df_google["idioma_padronizado"] = df_google["idioma"].apply(padronizar_idioma)
df_openalex["idioma_padronizado"] = df_openalex["idioma"].apply(padronizar_idioma)

#valores de idioma que ainda caem nao sao mapeados
for df, nome in [(df_google, "Google Scholar"), (df_openalex, "OpenAlex")]:
    nao_mapeados = df.loc[
        ~df["idioma_padronizado"].isin(list(MAP_IDIOMAS.values()) + ["Não informado"]), "idioma_padronizado"
    ].value_counts()
    if len(nao_mapeados):
        print(f"\n{nome}: valor(es) de idioma ainda sem mapeamento ->")
        print(nao_mapeados)

# Tipo documento

In [30]:
MAP_TIPO_DOCUMENTO = {
    "article": "Artigo",
    "conference-paper": "Artigo de Conferência",
    "dissertation": "Dissertação/Tese",
    "preprint": "Preprint",
    "dataset": "Dataset",
}

def padronizar_tipo_documento(valor):
    if pd.isna(valor) or str(valor).strip() == "":
        return "Não informado"

    chave = str(valor).strip().lower()
    return MAP_TIPO_DOCUMENTO.get(chave, str(valor).strip().title())


df_openalex["tipo_documento"] = df_openalex["tipo_documento"].apply(padronizar_tipo_documento)

nao_mapeados = df_openalex.loc[
    ~df_openalex["tipo_documento"].isin(list(MAP_TIPO_DOCUMENTO.values()) + ["Não informado"]),
    "tipo_documento"
].value_counts()
if len(nao_mapeados):
    print("OpenAlex: tipo(s) de documento ainda sem mapeamento ->")
    print(nao_mapeados)

print(f"\n{df_openalex['tipo_documento'].value_counts(dropna=False)}")


Series([], Name: count, dtype: int64)


# Integração Google Scholar + OpenAlex

In [31]:
df_google["fonte_dados"] = "Google Scholar"
df_openalex["fonte_dados"] = "OpenAlex"

df_citacoes = pd.concat([df_google, df_openalex], ignore_index=True, sort=False)

print(f"Total de registros antes da deduplicação: {len(df_citacoes):,}")

contagem_fontes = df_citacoes.groupby("id_citante")["fonte_dados"].agg(lambda x: "; ".join(sorted(set(x))))
print(contagem_fontes.value_counts())

apenas_uma_fonte = contagem_fontes[~contagem_fontes.str.contains(";")]
print(f"\nCitantes presentes em apenas uma fonte: {len(apenas_uma_fonte)} "
      f"(provavelmente DOI x título/ano não bateram entre as fontes)")

def combinar_fontes(series):
    return "; ".join(sorted(set(series.dropna().astype(str))))

def primeiro_valor(series):
    valores = series.dropna()
    valores = valores[valores.astype(str).str.strip() != ""]
    if len(valores) == 0:
        return np.nan
    return valores.iloc[0]

colunas_existentes = [c for c in COLUNAS_CONSOLIDADAS if c in df_citacoes.columns]

df_final = (
    df_citacoes
    .groupby("id_citante", as_index=False)
    .agg({
        **{coluna: primeiro_valor for coluna in colunas_existentes},
        "fonte_dados": combinar_fontes
    })
)

print(f"Registros após consolidação: {len(df_final):,}")
display(df_final[["id_geoinfo", "id_citante", "titulo", "ano", "fonte_dados"]].head(10))


print("IDs de citantes duplicados:", df_final["id_citante"].duplicated().sum())
print("Relações GEOINFO → citante duplicadas:", df_final[["id_geoinfo", "id_citante"]].duplicated().sum())


duplicados_id_geoinfo = df_geoinfo["identificador"].duplicated().sum()
if duplicados_id_geoinfo:
    print(f"Atenção: {duplicados_id_geoinfo} identificador(es) GEOINFO duplicado(s) "
          f"— mapa_ano_geoinfo vai manter só a última ocorrência")

mapa_ano_geoinfo = df_geoinfo.set_index("identificador")["ano"]
df_final["ano_geoinfo"] = df_final["id_geoinfo"].map(mapa_ano_geoinfo)

inconsistencias_temporais = df_final[df_final["ano"] < df_final["ano_geoinfo"]]
print(f"\nCitações anteriores ao artigo citado: {len(inconsistencias_temporais)}")


def categorizar_inconsistencia(row):
    if row["ano"] < 1990:
        return "ano_citante_implausivel"
    if not any(c.isalpha() for c in str(row["titulo"])[:3]) or len(str(row["titulo"]).split()) <= 4:
        return "titulo_suspeito_fragmento"
    return "revisar_manualmente"


inconsistencias_temporais = inconsistencias_temporais.copy()
inconsistencias_temporais["categoria"] = inconsistencias_temporais.apply(categorizar_inconsistencia, axis=1)

print(inconsistencias_temporais["categoria"].value_counts())
display(inconsistencias_temporais[["id_geoinfo", "id_citante", "titulo", "ano", "ano_geoinfo", "categoria"]])

Total de registros antes da deduplicação: 2,988
fonte_dados
Google Scholar    2208
Name: count, dtype: int64

Citantes presentes em apenas uma fonte: 2208 (provavelmente DOI x título/ano não bateram entre as fontes)
Registros após consolidação: 2,208


,id_geoinfo,id_citante,titulo,ano,fonte_dados
0,83LX3pFwXQZ3V9uMbiY/MdJMR,0018 2009 indexacao em bancos de dados espacia...,0018 2009 indexacao em bancos de dados espaciais,2009,Google Scholar
1,83LX3pFwXQZ3V9uMbiY/MdHjv,10 disseminacao de dados geograficos na intern...,10 disseminacao de dados geograficos na internet,2005,Google Scholar
2,83LX3pFwXQZ3V9uMbiY/Mekki,11 o open geospatial consortium|2005,11 o open geospatial consortium,2005,Google Scholar
3,83LX3pFwXQZ3V9uMbiY/Mipec,12 descricao da terralib|2005,12 descricao da terralib,2005,Google Scholar
4,83LX3pFwXQZ4BFjAq/mGafE,13 tratamento de dados matriciais na terralib|...,13 tratamento de dados matriciais na terralib,2005,Google Scholar
5,8JMKD3MGPDW34P/3SEURDB,3 resultados e discussao|2011,3 resultados e discussao,2011,Google Scholar
6,8JMKD3MGP8W/3FCBR3B,3d webgis from visualization to analysis an ef...,3d webgis from visualization to analysis an ef...,2018,Google Scholar
7,83LX3pFwXQZ3V9uMbiY/LHUxR,4d open spatial information infrastructure sup...,4d open spatial information infrastructure sup...,2019,Google Scholar
8,8JMKD3MGPDW34R/3UFE9F8,a 60 ghz millimeter wave fmcw radar system for...,a 60 ghz millimeter wave fmcw radar system for...,2025,Google Scholar
9,83LX3pFwXQZ3V9uMbiY/MipaK,a abordagem poesia para a integracao de dados ...,a abordagem poesia para a integracao de dados ...,2003,Google Scholar


IDs de citantes duplicados: 0
Relações GEOINFO → citante duplicadas: 0

Citações anteriores ao artigo citado: 18
categoria
revisar_manualmente          13
ano_citante_implausivel       4
titulo_suspeito_fragmento     1
Name: count, dtype: int64


,id_geoinfo,id_citante,titulo,ano,ano_geoinfo,categoria
5,8JMKD3MGPDW34P/3SEURDB,3 resultados e discussao|2011,3 resultados e discussao,2011,2018,titulo_suspeito_fragmento
273,8JMKD3MGP3W34P/42H35D2,an ontological analysis of observation collect...,an ontological analysis of observation collect...,2011,2014,revisar_manualmente
380,83LX3pFwXQZ3V9uMbiY/MdHjv,arbeitsgruppe automation in der kartographie t...,arbeitsgruppe automation in der kartographie t...,2001,2004,revisar_manualmente
446,83LX3pFwXQZ3V9uMbiY/Nib6T,b 12 sistemas de informacion geografica|2002,b 12 sistemas de informacion geografica,2002,2006,revisar_manualmente
487,8JMKD3MGPDW34P/4ADCP7P,centralidade e distribuicao espacial do comerc...,centralidade e distribuicao espacial do comerc...,2021,2023,revisar_manualmente
512,8JMKD3MGP3W34P/42GRQBL,cluster lifecycle analysis challenges techniqu...,cluster lifecycle analysis challenges techniqu...,1901,2014,ano_citante_implausivel
838,8JMKD3MGPDW34P/4ADCN3L,evidence for rice tolerance to tibraca limbati...,evidence for rice tolerance to tibraca limbati...,2021,2023,revisar_manualmente
855,83LX3pFwXQZ3V9uMbiY/LHU82,exploratory spatial analisys of criminal data|...,exploratory spatial analisys of criminal data,2001,2005,revisar_manualmente
897,8JMKD3MGPDW34P/4ADCN3L,field responses of rice stalk stink bug tibrac...,field responses of rice stalk stink bug tibrac...,2021,2023,revisar_manualmente
908,8JMKD3MGPDW34P/4ADCN3L,forecasting grain production and static capaci...,forecasting grain production and static capaci...,2021,2023,revisar_manualmente


# Resumo dos dados coletados e integrados

In [32]:
relatorio = pd.DataFrame({
    "Etapa": [
        "Artigos GEOINFO",
        "Registros Google Scholar",
        "Registros OpenAlex",
        "Registros de citações antes da integração",
        "Trabalhos citantes após consolidação",
        "Citantes presentes em ambas as fontes",
        "Citantes presentes em apenas uma fonte",
        "Citações com inconsistência temporal (ano < ano do artigo citado)",
    ],

    "Quantidade": [
        len(df_geoinfo),
        len(df_google),
        len(df_openalex),
        len(df_citacoes),
        len(df_final),
        (contagem_fontes.str.contains(";")).sum(),
        len(apenas_uma_fonte),
        len(inconsistencias_temporais),
    ]
})

display(relatorio)

resumo_fontes = (
    df_final["fonte_dados"]
    .value_counts()
    .rename_axis("fonte")
    .reset_index(name="quantidade")
)

display(resumo_fontes)

,Etapa,Quantidade
0,Artigos GEOINFO,681
1,Registros Google Scholar,2988
2,Registros OpenAlex,0
3,Registros de citações antes da integração,2988
4,Trabalhos citantes após consolidação,2208
5,Citantes presentes em ambas as fontes,0
6,Citantes presentes em apenas uma fonte,2208
7,Citações com inconsistência temporal (ano < an...,18


,fonte,quantidade
0,Google Scholar,2208


# Salvar resultados

In [33]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

nulos_criticos = df_final[["id_geoinfo", "titulo"]].isna().sum()
if nulos_criticos.sum() > 0:
    print(f"Atenção: valores nulos em colunas críticas antes de salvar ->\n{nulos_criticos}")

arquivos = {
    "trabalhos_geoinfo_final.csv": df_geoinfo,
    "trabalhos_citantes_final.csv": df_final,
    "autores_instituicoes_geoinfo.csv": df_autores_instituicoes_geoinfo,
    "relatorio_qualidade_dados.csv": relatorio,
    "resumo_fontes.csv": resumo_fontes,
}

for nome_arquivo, df in arquivos.items():
    caminho = OUTPUT_DIR / nome_arquivo
    df.to_csv(caminho, index=False, encoding="utf-8-sig")
    print(f"{nome_arquivo}: {len(df):,} linha(s) salva(s)")

print(f"\nTodos os arquivos salvos em: {OUTPUT_DIR.resolve()}")

trabalhos_geoinfo_final.csv: 681 linha(s) salva(s)
trabalhos_citantes_final.csv: 2,208 linha(s) salva(s)
autores_instituicoes_geoinfo.csv: 2,619 linha(s) salva(s)
relatorio_qualidade_dados.csv: 8 linha(s) salva(s)
resumo_fontes.csv: 1 linha(s) salva(s)

Todos os arquivos salvos em: C:\Users\giuli\OneDrive\Área de Trabalho\INPE\ciencia de dados\citacoes_geoinfo\data\processed


In [34]:
print(df_final.columns.tolist())

['id_geoinfo', 'id_citante', 'titulo', 'ano', 'autores', 'instituicoes', 'idioma_padronizado', 'pais_padronizado', 'veiculo_publicacao', 'doi_normalizado', 'url', 'tipo_documento', 'topico', 'subcampo', 'campo', 'dominio_tematico', 'fonte_publicacao', 'status_acesso_aberto', 'openalex_id', 'fonte_dados', 'ano_geoinfo']
